# Practical Machine Learning: scikit-learn + PySpark
### A hands-on course: choose models, understand them, tune them, and know when they fail

**Audience:** comfortable with Python/pandas; new to systematic ML model selection. **Time:** ~3–5 hours, split into sessions. **Run order:** top to bottom. Sections 1–7 work without Spark; section 8 uses Java + PySpark.

**Objectives:** compare sensible baselines; explain linear/logistic models, kNN, trees, random forests, boosting, SVM, Naive Bayes and clustering; diagnose data leakage, overfitting, underfitting, class imbalance, useless predictions and distribution shift; tune without contaminating the test set; use Spark ML pipelines on a laptop.

This is executable teaching material, not a claim that the "best model" can be chosen without data. Each exercise asks you to predict an outcome *before* running the next cell.

## 0. Setup and safe execution
From your project folder, install with **uv** in a terminal:
```bash
uv add numpy pandas matplotlib scikit-learn scipy jupyterlab
uv add pyspark
uv run jupyter lab
```
Choose this project's Python environment as the notebook kernel. Current PySpark 4.x requires **Python ≥3.10 and Java ≥17**; check `java -version` in a terminal. Spark sections run with `local[2]` (two cores) and are kept small. **Do not use `collect()` or `toPandas()` on unbounded datasets.**

No internet or API keys are needed after package installation. Plots are produced as you run the cells. All synthetic randomness is seeded so results can be repeated.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification, make_regression, make_moons
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score, silhouette_score)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
SEED = 42
np.random.seed(SEED)
print('Python:', sys.version.split()[0], '| sklearn:', sklearn.__version__)
plt.rcParams['figure.figsize'] = (7, 4)

## 1. The model-selection map
**First identify your prediction task and evaluation constraint.**

| Problem | Good first models | When to try another | Primary metrics |
|---|---|---|---|
| Predict a continuous number | Dummy mean, Linear/Ridge, RandomForestRegressor, HistGradientBoostingRegressor | Nonlinear effects, interactions | MAE, RMSE, R² |
| Predict one of two classes | Dummy majority, LogisticRegression, decision tree, random forest, histogram boosting | Nonlinear boundaries, imbalance | Precision/recall/F1, ROC-AUC, average precision |
| Many classes | LogisticRegression, random forest, boosting | Different costs per class | Macro F1, confusion matrix |
| Group unlabeled examples | KMeans, DBSCAN | Clusters aren't spherical / noise | Visual inspection, stability, silhouette (with care) |
| Text classification | TF-IDF + linear model / Naive Bayes | Complex semantics | F1, precision/recall |
| Forecast future values | Seasonal naive, lag-feature linear/tree models | Drift / seasonality | Time-ordered MAE + backtesting |

**Trade-offs:** Logistic/linear = interpretable coefficients and fast, but limited unless features capture nonlinearities. kNN = distance-based, needs scaling, suffers in high dimensions. Decision tree = readable rules but unstable/high-variance. Random forest = many bootstrapped, decorrelated trees to reduce variance. Gradient boosting = sequential trees that correct mistakes; often strong for tabular data, but tuning matters. SVM = margin-based separator, kernel can form nonlinear boundaries but scaling and compute matter. Naive Bayes = probabilistic conditional-independence assumption, often strong on sparse text. Neural networks = typically more compelling for images/audio/text or plentiful data, not automatically best for small tables.

**Algorithm is only one decision.** Ask: what information exists *at prediction time*? Are observations independent? Is the cost of a false positive different from a false negative? What score would beat doing nothing?

## 2. Classification experiment: the identical dataset for every model
We'll simulate a moderately imbalanced binary prediction problem. A `1` can be thought of as a rare event. We separate a **test set once** and never use it to choose hyperparameters. Stratification preserves the class ratio; it does not protect against time or entity leakage.

In [ ]:
X, y = make_classification(n_samples=2400, n_features=12, n_informative=5,
    n_redundant=3, weights=[0.84, 0.16], class_sep=0.9, flip_y=0.04, random_state=SEED)
features = [f'x{i}' for i in range(X.shape[1])]
X = pd.DataFrame(X, columns=features)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
    stratify=y, random_state=SEED)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Class fractions:', pd.Series(y_train).value_counts(normalize=True).sort_index().round(3).to_dict())
pd.Series(y_train).value_counts().sort_index().plot(kind='bar', title='Training labels: 0 vs 1')
plt.xlabel('Class'); plt.ylabel('Count'); plt.show()

### How bad is doing nothing?
A **dummy baseline** ignores the features and always predicts the most frequent class. Its accuracy can be deceptively high on imbalanced data. For a rare class, inspect **recall** (fraction of actual positives found) and **precision** (fraction of predicted positives that are correct) alongside accuracy. ROC-AUC describes ranking discrimination, not the quality of a specific decision threshold.

In [ ]:
models = {
    'Dummy majority': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1500, random_state=SEED))]),
    'kNN': Pipeline([('scale', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=15))]),
    'Decision tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=SEED),
    'Random forest': RandomForestClassifier(n_estimators=150, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    'Gradient boosting': HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, random_state=SEED),
    'SVM (RBF)': Pipeline([('scale', StandardScaler()), ('model', SVC(probability=True, random_state=SEED))]),
    'Gaussian NB': GaussianNB()
}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv,
        scoring={'balanced_accuracy': 'balanced_accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
        n_jobs=1)
    results.append({'Model':name, 'Balanced accuracy':np.mean(scores['test_balanced_accuracy']),
                    'F1':np.mean(scores['test_f1']), 'ROC-AUC':np.mean(scores['test_roc_auc']),
                    'F1 SD':np.std(scores['test_f1'])})
scoreboard = pd.DataFrame(results).sort_values('F1', ascending=False)
display(scoreboard.round(3))
ax = scoreboard.sort_values('F1').plot.barh(x='Model', y=['F1','ROC-AUC','Balanced accuracy'], figsize=(10,6))
ax.set_title('4-fold cross-validation on TRAIN data only (higher is better)')
ax.set_xlim(0, 1); plt.tight_layout(); plt.show()

**Interpretation:** We compare CV scores before opening the test set. A stable model has similar fold scores (small standard deviation). Select according to business cost, not which bar is tallest by 0.003. Models with more complexity aren't automatically more useful.

## 3. Decision trees, explained in detail
A tree repeatedly asks **yes/no questions**, such as `x3 <= 0.4`. Each split is chosen to make child nodes more homogeneous in label. Common classification criteria: **Gini impurity** `1 − Σ p(class)²` or entropy `−Σ p log₂ p`. A pure node has impurity zero. Each leaf predicts using training examples that reach it.

`max_depth` caps how many sequential decisions can be made; `min_samples_leaf` prevents tiny leaves. **Deep unrestricted trees can memorize noise.** Their predictions are piecewise constant and can shift when the training data changes slightly. Trees typically don't need feature scaling. A tree's displayed thresholds are in the units of its input data.

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED)
small_tree.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(15, 7))
plot_tree(small_tree, feature_names=features, class_names=['0','1'],
          filled=True, rounded=True, fontsize=8, ax=ax, impurity=True, proportion=True)
ax.set_title('Read top-down: a branch is a rule; a leaf is a prediction')
plt.show()
print('Train accuracy:', round(small_tree.score(X_train, y_train),3))
print('Test accuracy:', round(small_tree.score(X_test, y_test),3), '(illustration only; do not use to tune)')

### Overfitting and underfitting: compare train with validation
- **Underfitting:** model too simple; both training and validation scores poor.
- **Overfitting:** training score great, validation considerably worse; more depth often creates this.
- **Healthy fit:** good validation performance and a modest train–validation gap.

Below, we vary only tree depth using **cross-validation on training data**, not the held-out test set.

In [ ]:
depths = [1,2,3,4,5,7,10,15,None]
train_s, val_s = [], []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    cv_scores = cross_validate(tree, X_train, y_train, cv=cv,
                               scoring='balanced_accuracy', return_train_score=True)
    train_s.append(cv_scores['train_score'].mean())
    val_s.append(cv_scores['test_score'].mean())
plt.plot(range(len(depths)), train_s, marker='o', label='Training')
plt.plot(range(len(depths)), val_s, marker='o', label='Validation (CV)')
plt.xticks(range(len(depths)), [str(d) for d in depths]); plt.xlabel('Maximum tree depth')
plt.ylabel('Balanced accuracy'); plt.title('Diagnose capacity and overfitting'); plt.legend(); plt.grid(alpha=.2); plt.show()
print('Best validation depth in this plot:', depths[int(np.argmax(val_s))])

## 4. What each model *actually learns*: 2D decision boundaries
Synthetic two-moons points form an intentionally nonlinear pattern. Watch linear logistic regression struggle to form a curved boundary, while kNN/trees/forests can represent it. **Illustration only:** the toy pattern doesn't prove which model wins on real datasets.

In [ ]:
Xm, ym = make_moons(n_samples=550, noise=0.28, random_state=SEED)
toy_models = {
 'Logistic': Pipeline([('scale',StandardScaler()),('m',LogisticRegression())]),
 'kNN': Pipeline([('scale',StandardScaler()),('m',KNeighborsClassifier(n_neighbors=15))]),
 'Tree depth 3': DecisionTreeClassifier(max_depth=3, random_state=SEED),
 'Random forest': RandomForestClassifier(n_estimators=80, max_depth=6, random_state=SEED)
}
xx, yy = np.meshgrid(np.linspace(-2,3,160), np.linspace(-1.5,2,160))
fig, axes = plt.subplots(2, 2, figsize=(10,8))
for ax,(name, clf) in zip(axes.flat, toy_models.items()):
    clf.fit(Xm,ym)
    zz = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,zz,levels=[-.5,.5,1.5],alpha=.25,cmap='coolwarm')
    ax.scatter(Xm[:,0], Xm[:,1], c=ym, s=8, cmap='coolwarm', alpha=.7)
    ax.set_title(name)
fig.suptitle('Different learned decision boundaries'); plt.tight_layout(); plt.show()

## 5. Evaluation beyond accuracy
A **confusion matrix** counts true positives, false positives, true negatives, and false negatives. Threshold choice changes this tradeoff. For risk-screening, missing a positive may be worse than falsely flagging one; for expensive interventions, false positives may be costly. Neither F1 nor ROC-AUC knows your actual costs.

**Important:** here we *illustrate* thresholds with test observations. In a real project, optimize the threshold on validation predictions only, then evaluate it once on test data.

In [ ]:
clf = Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=1500))])
clf.fit(X_train,y_train)
prob = clf.predict_proba(X_test)[:,1]
fig, ax = plt.subplots(1,2,figsize=(11,4))
ConfusionMatrixDisplay.from_predictions(y_test, prob>=0.5, ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test,prob,ax=ax[1]); plt.tight_layout(); plt.show()
rows=[]
for t in [0.2,0.35,0.5,0.7]:
    pred = (prob>=t).astype(int)
    rows.append({'Threshold':t,'Precision':precision_score(y_test,pred,zero_division=0),
                 'Recall':recall_score(y_test,pred,zero_division=0),
                 'F1':f1_score(y_test,pred,zero_division=0),
                 'Predicted positives':int(pred.sum())})
display(pd.DataFrame(rows).round(3))

### Exercise: threshold selection without test leakage
Make `X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, stratify=y_train, random_state=SEED, test_size=.2)`. Fit on `X_fit`, pick threshold on `X_val` using a stated cost function such as `5 × FN + 1 × FP`, retrain on full training set, and apply your chosen threshold once to test. **Do not** pick the threshold from the table above for a final report.

## 6. Hyperparameter optimization: systematic rather than manual
**Parameters** are learned from data (tree thresholds, logistic coefficients). **Hyperparameters** are settings you choose before fitting (`max_depth`, `C`, `min_samples_leaf`).

`GridSearchCV` trains each combination across validation folds. Search multiple candidates but avoid enormous grids. `RandomizedSearchCV` samples configurations and can be more efficient in wide search spaces. **Pipelines** keep preprocessing fitted inside each fold, preventing validation leakage. We optimize *balanced accuracy* here because classes are uneven; other problems require different targets.

In [ ]:
search_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(random_state=SEED))
])
param_grid = {'model__max_depth':[2,4,6,9,None],
              'model__min_samples_leaf':[2,10,30]}
grid = GridSearchCV(search_pipeline, param_grid=param_grid, scoring='balanced_accuracy',
                    cv=cv, n_jobs=-1, return_train_score=True)
grid.fit(X_train,y_train)
print('Best CV settings:',grid.best_params_)
print('Best mean CV balanced accuracy:',round(grid.best_score_,3))
cv_results = pd.DataFrame(grid.cv_results_)
visual = cv_results.pivot_table(index='param_model__min_samples_leaf',
    columns='param_model__max_depth',values='mean_test_score')
visual = visual.reindex(index=[2,10,30], columns=[2,4,6,9,None])
fig,ax=plt.subplots(figsize=(8,4))
im=ax.imshow(visual.astype(float),aspect='auto',vmin=.5,vmax=1,cmap='viridis')
ax.set_xticks(range(len(visual.columns)),[str(v) for v in visual.columns])
ax.set_yticks(range(len(visual.index)),[str(v) for v in visual.index])
ax.set_xlabel('max_depth');ax.set_ylabel('min_samples_leaf');ax.set_title('CV balanced accuracy: tree tuning')
fig.colorbar(im,ax=ax);plt.show()

### Honest final test (once, after selection)
The grid's `best_score_` is **not an unbiased future-performance estimate**, because choosing the maximum exploits CV noise. We reserved an untouched test set. Report test metrics along with uncertainty if decisions are important; repeated test-set use in model iteration turns it into another validation set.

In [ ]:
final_tree = grid.best_estimator_
test_pred = final_tree.predict(X_test)
print(pd.Series({'Test balanced accuracy':balanced_accuracy_score(y_test,test_pred),
                 'Test precision':precision_score(y_test,test_pred,zero_division=0),
                 'Test recall':recall_score(y_test,test_pred,zero_division=0),
                 'Test F1':f1_score(y_test,test_pred,zero_division=0)}).round(3))
ConfusionMatrixDisplay.from_predictions(y_test,test_pred,colorbar=False);plt.show()

## 7. Diagnostic lab: when a model is *broken* or doing nothing
| Symptom | Likely cause | Test / fix |
|---|---|---|
| 85% accuracy, 0% recall | Majority-class predictions | Compare dummy; confusion matrix; PR curve; rebalance/cost weighting if justified |
| Train 99%, CV 60% | Overfitting or leakage | Reduce complexity; scrutinize feature provenance; group/time split |
| Train 55%, CV 54% | Underfitting, noise, invalid features | Better features, different family, label audit |
| CV 99%, real-world 55% | Leakage, duplicates, distribution shift | Time/entity-aware split; audit feature timestamps |
| Predicts identical number for everyone | Poor signal, excessive regularization, failed feature pipeline | Check prediction variance, feature variance, baseline |
| Slow fit, no improvement | High-cardinality data / too many trials | Subsample, reduce features, fewer CV folds, tune budget |
| Good ranking, bad threshold outcomes | Threshold / class prevalence mismatch | Calibrate and tune threshold on validation only |
| Very different subgroup scores | Heterogeneity, sparse segments, possible unfairness | Slice metrics; obtain more representative data |

**Checklist before deployment:** valid target and timestamps → no duplicate entity crossing split → baseline → cross-validation chosen for real deployment → metric aligned with costs → feature importance / ablation → subgroup checks → held-out final test → drift monitoring.

In [ ]:
# Demonstration: intentionally broken majority-class predictor
majority = DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
p0 = majority.predict(X_test)
print('Accuracy:',round(accuracy_score(y_test,p0),3),'Recall:',round(recall_score(y_test,p0,zero_division=0),3))
print('Unique predictions:',np.unique(p0,return_counts=True))
ConfusionMatrixDisplay.from_predictions(y_test,p0,colorbar=False);plt.show()
# Permutation importance: how much held-out score drops when a feature is shuffled.
# We use this as an illustration, NOT a tuning criterion. Correlated features can split importance.
imp = permutation_importance(final_tree, X_test, y_test, n_repeats=5,
                             scoring='balanced_accuracy', random_state=SEED)
pd.Series(imp.importances_mean,index=features).sort_values().plot.barh(title='Permutation importance (test illustration)')
plt.xlabel('Mean balanced-accuracy drop after shuffling');plt.show()

### Leakage challenge
Suppose a dataset contains `refund_issued` and the goal is to predict `will_refund`. If the `refund_issued` column only becomes known *after* the prediction time, it is a **leaky feature**. Even 99% validation accuracy is misleading if that information would be unavailable at inference time. Likewise, same customer's records across random folds can inflate CV results. Use `GroupKFold` for unseen customers, or time-ordered splits for forecasting; never randomly shuffle future events into the past.

## 8. Regression: numbers, not categories
**LinearRegression** finds coefficients minimizing squared residuals, i.e. prediction error. **Ridge** adds L2 regularization that shrinks coefficients (often useful with correlated features). Decision-tree-based regression captures nonlinear effects and interactions but may not extrapolate beyond seen targets. MAE = average absolute error in target units; RMSE penalizes large errors more; R² compares with predicting the mean and can be negative.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
Xr, yr = make_regression(n_samples=1400,n_features=10,n_informative=5,
                         noise=25,random_state=SEED)
Xra, Xrb, yra, yrb = train_test_split(Xr,yr,test_size=.25,random_state=SEED)
regressors = {'Mean baseline':DummyRegressor(),
    'Linear':LinearRegression(),
    'Ridge':Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=10))]),
    'Forest':RandomForestRegressor(n_estimators=100,min_samples_leaf=3,n_jobs=-1,random_state=SEED)}
reg_rows=[]
for name, reg in regressors.items():
    reg.fit(Xra,yra);yp=reg.predict(Xrb)
    reg_rows.append({'Model':name,'MAE':mean_absolute_error(yrb,yp),
                     'RMSE':np.sqrt(mean_squared_error(yrb,yp)), 'R²':r2_score(yrb,yp)})
display(pd.DataFrame(reg_rows).round(2))
reg = regressors['Ridge']; predicted=reg.predict(Xrb)
plt.scatter(yrb,predicted,s=10,alpha=.5)
lims=[min(yrb.min(),predicted.min()),max(yrb.max(),predicted.max())]
plt.plot(lims,lims,'k--');plt.xlabel('Actual');plt.ylabel('Predicted');plt.title('Residual spread around perfect-prediction line');plt.show()

## 9. PySpark fundamentals: distributed DataFrames + MLlib
**scikit-learn** fits models in one Python process (though algorithms can parallelize on one machine). **PySpark** uses Spark's DataFrame execution engine and can distribute data processing and model fitting across workers. Spark is not automatically faster on a small laptop dataset: startup, serialization, scheduling and shuffle overhead may dominate. Spark ML uses `features` as a **single vector column** and `label` as a numeric column.

Spark is **lazy**: `.filter()`/`.select()` describe work; `.count()`/`.show()` trigger execution (**actions**). Prefer built-in column functions over Python UDFs for simple operations.

In [ ]:
# Separate cell so the entire scikit-learn course works even if Spark/Java isn't set up.
try:
    import pyspark
    from pyspark.sql import SparkSession, functions as F
    from pyspark.ml import Pipeline as SparkPipeline
    from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkStandardScaler
    from pyspark.ml.classification import LogisticRegression as SparkLogisticRegression
    from pyspark.ml.classification import DecisionTreeClassifier as SparkDecisionTree
    from pyspark.ml.classification import RandomForestClassifier as SparkRandomForest
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
    spark = (SparkSession.builder.master('local[2]').appName('MLLearningLab')
             .config('spark.sql.shuffle.partitions','4').getOrCreate())
    spark.sparkContext.setLogLevel('ERROR')
    SPARK_READY=True
    print('Spark version:',spark.version)
except Exception as exc:
    SPARK_READY=False
    print('Spark not available. Install pyspark and Java 17+, restart kernel, rerun this cell.')
    print(type(exc).__name__,str(exc)[:300])

In [ ]:
if SPARK_READY:
    spark_df = spark.createDataFrame(X_train.assign(label=y_train.astype(float)))
    print('Rows:',spark_df.count(),'Partitions:',spark_df.rdd.getNumPartitions())
    spark_df.select('x0','x1','label').show(5)
    spark_df.groupBy('label').count().orderBy('label').show()
    spark_df.filter(F.col('x0')>0).select('x0','label').show(5)
    spark_df.explain(mode='simple')
else:
    print('Skipping Spark: set up the environment in previous cell.')

### Spark ML Pipeline: assembling features, training, measuring
In Spark, `VectorAssembler` combines numeric columns into one vector. `Pipeline` ensures the transformations are reproduced for new records. Spark's random split is convenient for illustration; for real work **split by date, customer or other entity as appropriate**. For production class imbalance, confirm class proportions in both splits and use a stratified/entity split upstream when appropriate.

In [ ]:
if SPARK_READY:
    assembler = VectorAssembler(inputCols=features,outputCol='features',handleInvalid='error')
    spark_lr = SparkLogisticRegression(featuresCol='features',labelCol='label',maxIter=60,
                                       regParam=.05)
    spark_pipe = SparkPipeline(stages=[assembler,spark_lr])
    # We reuse the sklearn HOLDOUT split to avoid accidentally mixing partitions.
    spark_train=spark.createDataFrame(X_train.assign(label=y_train.astype(float)))
    spark_test=spark.createDataFrame(X_test.assign(label=y_test.astype(float)))
    spark_fit=spark_pipe.fit(spark_train)
    spark_predictions=spark_fit.transform(spark_test).cache()
    spark_predictions.select('label','prediction','probability').show(6,truncate=False)
    spark_auc=BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC').evaluate(spark_predictions)
    spark_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(spark_predictions)
    print('Spark ROC-AUC:',round(spark_auc,3),'Spark weighted F1:',round(spark_f1,3))
    print('Note: Spark weighted F1 is NOT sklearn positive-class F1.')
else:
    print('Spark section skipped.')

### Tune a Spark tree using cross-validation **within training data**
Spark `CrossValidator` evaluates each parameter combination across folds. To save laptop compute, this experiment uses 3 folds and 4 candidate settings. We use **areaUnderROC**, which compares rankings; also inspect recall and precision for the final decision threshold. Spark's CV default fold construction is random, not group/time aware; do not use blindly with temporally dependent data.

In [ ]:
if SPARK_READY:
    tree_est = SparkDecisionTree(labelCol='label',featuresCol='features',seed=SEED)
    tree_pipe = SparkPipeline(stages=[assembler,tree_est])
    parameter_grid = (ParamGridBuilder().addGrid(tree_est.maxDepth,[3,6])
                      .addGrid(tree_est.minInstancesPerNode,[5,25]).build())
    evaluator = BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC')
    spark_cv = CrossValidator(estimator=tree_pipe,estimatorParamMaps=parameter_grid,
                             evaluator=evaluator,numFolds=3,seed=SEED,parallelism=2)
    spark_cv_fit=spark_cv.fit(spark_train)
    best_spark_tree=spark_cv_fit.bestModel.stages[-1]
    print('Best Spark tree maxDepth:',best_spark_tree.getMaxDepth())
    print('Best Spark tree minInstancesPerNode:',best_spark_tree.getMinInstancesPerNode())
    print('Train-CV AUC by candidate:',np.round(spark_cv_fit.avgMetrics,3))
    print('Held-out test AUC:',round(evaluator.evaluate(spark_cv_fit.transform(spark_test)),3))
else:
    print('Spark tuning skipped.')

### Spark diagnostics / avoiding painful mistakes
- `.show(5)` and `.limit(1000).toPandas()` for inspection; **never** convert entire big data to pandas just to plot.
- `.explain()` to inspect execution plans. Watch for wide shuffles (`Exchange`) and skew.
- `.cache()` only when you reuse an expensive DataFrame; `.unpersist()` afterward.
- **Local mode `local[2]` is not a multi-machine cluster.** It's for learning the API, not a speed benchmark.
- Feature scaling: use Spark `StandardScaler` for scale-sensitive learners (LR/SVM/k-means), but *trees generally do not need it*. Some Spark estimators have internal standardization options.
- When scoring millions of rows, aggregate metrics in Spark rather than `collect()` predictions.
- Spark `.randomSplit()` is not a safe time-series split or guaranteed stratified split.

**Stopping Spark when done:**

In [ ]:
if SPARK_READY:
    spark.stop()
    print('Spark session stopped. Restart Spark setup cell to use Spark again.')

## 10. Mini-project and lab exercises
**Project A (classification):** Replace the toy data with a real tabular target, or make a synthetic churn dataset with `make_classification`. Write down a prediction-time schema. Report majority baseline, logistic regression, tree, forest, and gradient boosting. Tune one family via 4-fold CV, then open test once. Explain top errors, not just score.

**Project B (regression):** Add nonlinear signal: `yr2 = yr + 60 * (Xr[:,0] > 0)`. Compare Ridge and forest on the same train/test split. Plot residuals against a feature and explain where a linear model fails.

**Project C (Spark):** Increase rows from 2,400 to 50,000 using `make_classification`; create Spark DataFrame from pandas only at this toy scale. Compare Spark LR and Spark RF. Measure startup/fit durations; do *not* expect Spark to beat sklearn on one laptop.

**Challenge questions (answer before inspecting metrics):**
1. Accuracy = 94%, positive-class recall = 0%. Is the model useful for detecting rare events? Why?
2. CV score rises as `max_depth` goes 2→4→6, but falls after 10. What would you tune next?
3. Which column could encode information not available at inference time?
4. You tested 30 configurations on the held-out test set and chose the winner. Is it held out anymore?
5. Does ROC-AUC = 0.88 mean 88% of decisions are correct? Explain.
6. When would a time split beat random stratified folds? When would GroupKFold be necessary?

**Answers:** 1: Likely no for positive detection; majority baseline can yield high accuracy and no positives. 2: Depth, `min_samples_leaf`, maybe a different model; CV and stability first. 3: Any post-outcome feature (e.g. refund paid), future aggregate or label proxy. 4: No, the test is contaminated through model selection. 5: No; AUC is the probability a random positive outranks a random negative (assuming standard conditions), not classification accuracy. 6: Time-ordered deployment and repeat entities/customers, respectively.

## References / continued study (primary documentation)
- [scikit-learn: supervised learning methods](https://scikit-learn.org/stable/supervised_learning.html)
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [scikit-learn: model evaluation metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn: GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [Apache Spark: PySpark installation and Java requirements](https://spark.apache.org/docs/latest/api/python/getting_started/install.html)
- [Apache Spark: Spark ML API (pipelines, estimators, evaluators, CrossValidator)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html)

**What this notebook deliberately does not teach fully:** formal probability calibration, nested cross-validation, probabilistic uncertainty, neural network training, production feature stores, and cluster deployment. Those are reasonable next modules after mastering the diagnostics above.